<a href="https://colab.research.google.com/github/digitalirrigation-lgtm/Gode-Water-Dashboard/blob/main/Gode_Irrigation_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import ee
ee.Authenticate()

In [3]:
ee.Initialize(project='irrigation-intelligence-pro')
print("Earth Engine is ready for Project 2!")

Earth Engine is ready for Project 2!


In [4]:
import ee, geemap
ee.Initialize(project='irrigation-intelligence-pro')

# 1. Load the GAUL 2025 boundary dataset
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')

# 2. Filter step-by-step for Gode Zone
gode = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
gode = gode.filter(ee.Filter.eq('GAUL1_NAME', 'Somali'))
gode = gode.filter(ee.Filter.eq('GAUL2_NAME', 'Gode'))

# 3. Extract the pure geometry
aoi_gode = gode.geometry()

# 4. Calculate the area to check if it is correct
area_sqkm = aoi_gode.area().divide(1000000).getInfo()
print(f"Gode Zone Area: {area_sqkm:,.0f} square kilometers")

Gode Zone Area: 0 square kilometers


In [5]:
import ee, geemap
ee.Initialize(project='irrigation-intelligence-pro')

# 1. Load the GAUL 2025 boundary dataset
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')

# 2. Filter for the correct zone name: "Shabelle" (previously Gode)
gode = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
gode = gode.filter(ee.Filter.eq('GAUL1_NAME', 'Somali'))
gode = gode.filter(ee.Filter.eq('GAUL2_NAME', 'Shabelle')) # <-- corrected name

# 3. Extract the pure geometry
aoi_gode = gode.geometry()

# 4. Calculate the area to check if it is correct
area_sqkm = aoi_gode.area().divide(1000000).getInfo()
print(f"Gode (Shabelle) Zone Area: {area_sqkm:,.0f} square kilometers")

Gode (Shabelle) Zone Area: 34,005 square kilometers


In [6]:
import ee
ee.Initialize(project='irrigation-intelligence-pro')

# 1. Rebuild the Gode (Shabelle) boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
gode = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
gode = gode.filter(ee.Filter.eq('GAUL1_NAME', 'Somali'))
gode = gode.filter(ee.Filter.eq('GAUL2_NAME', 'Shabelle'))
aoi_gode = gode.geometry()

# 2. Load CHIRPS Daily Rainfall
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')

# 3. Filter by Boundary, then by Date (Strict Algorithm Order)
chirps_filtered = chirps.filterBounds(aoi_gode)
chirps_filtered = chirps_filtered.filterDate('2021-01-01', '2025-12-31')

# 4. Count how many daily images we have
count = chirps_filtered.size().getInfo()
print(f"Number of daily rainfall images from 2021 to 2025: {count}")


Number of daily rainfall images from 2021 to 2025: 1825


In [7]:
import ee
ee.Initialize(project='irrigation-intelligence-pro')

# 1. Rebuild boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
gode = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
gode = gode.filter(ee.Filter.eq('GAUL1_NAME', 'Somali'))
gode = gode.filter(ee.Filter.eq('GAUL2_NAME', 'Shabelle'))
aoi_gode = gode.geometry()

# 2. Load and filter CHIRPS
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
chirps_filtered = chirps.filterBounds(aoi_gode).filterDate('2021-01-01', '2025-12-31')

# 3. Set up the month sequence (60 months total)
start_date = ee.Date('2021-01-01')
# 5 years * 12 months = 60. Sequence creates a list from 0 to 59.
months = ee.List.sequence(0, 59)

# 4. The function that sums the daily rain into one monthly image
def calculate_monthly_total(month_offset):
    # Find the start and end date for this specific month
    current_month_start = start_date.advance(month_offset, 'month')
    current_month_end = current_month_start.advance(1, 'month')

    # Filter the daily collection to just this month, and sum it
    monthly_rain = chirps_filtered.filterDate(current_month_start, current_month_end).sum()

    # Attach the date to the new image so we don't lose it
    return monthly_rain.set('system:time_start', current_month_start.millis())

# 5. Apply the function to all 60 months at once
monthly_chirps = ee.ImageCollection.fromImages(months.map(calculate_monthly_total))

# 6. Check how many monthly images we made
monthly_count = monthly_chirps.size().getInfo()
print(f"Number of monthly rainfall images created: {monthly_count}")

Number of monthly rainfall images created: 60


In [8]:
import ee
ee.Initialize(project='irrigation-intelligence-pro')

# Rebuild boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
gode = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
gode = gode.filter(ee.Filter.eq('GAUL1_NAME', 'Somali'))
gode = gode.filter(ee.Filter.eq('GAUL2_NAME', 'Shabelle'))
aoi_gode = gode.geometry()

# Load the REAL CHIRPS data
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')

# 1. Filter for the 30-year baseline (1991 to 2020)
baseline_chirps = chirps.filterBounds(aoi_gode).filterDate('1991-01-01', '2020-12-31')

# 2. Set up the 30-year month sequence (30 years * 12 months = 360 months)
baseline_start = ee.Date('1991-01-01')
baseline_months = ee.List.sequence(0, 359)

# 3. The function to sum daily rain into monthly totals
def calculate_baseline_monthly(month_offset):
    current_month_start = baseline_start.advance(month_offset, 'month')
    current_month_end = current_month_start.advance(1, 'month')
    monthly_rain = baseline_chirps.filterDate(current_month_start, current_month_end).sum()
    return monthly_rain.set('system:time_start', current_month_start.millis())

# 4. Apply the function
baseline_monthly_chirps = ee.ImageCollection.fromImages(baseline_months.map(calculate_baseline_monthly))

print(f"Baseline monthly images created: {baseline_monthly_chirps.size().getInfo()}")

Baseline monthly images created: 360


In [10]:
import ee
ee.Initialize(project='irrigation-intelligence-pro')

# 1. Rebuild boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
gode = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
gode = gode.filter(ee.Filter.eq('GAUL1_NAME', 'Somali'))
gode = gode.filter(ee.Filter.eq('GAUL2_NAME', 'Shabelle'))
aoi_gode = gode.geometry()

# 2. Load and filter CHIRPS
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')

# Create the Baseline (1991-2020)
baseline_chirps = chirps.filterBounds(aoi_gode).filterDate('1991-01-01', '2020-12-31')
baseline_months = ee.List.sequence(0, 359)
def calc_baseline(offset):
    start = ee.Date('1991-01-01').advance(offset, 'month')
    end = start.advance(1, 'month')
    rain = baseline_chirps.filterDate(start, end).sum()
    return rain.set('system:time_start', start.millis())
baseline_monthly = ee.ImageCollection.fromImages(baseline_months.map(calc_baseline))

# Create the Study Period (2021-2025)
study_chirps = chirps.filterBounds(aoi_gode).filterDate('2021-01-01', '2025-12-31')
study_months = ee.List.sequence(0, 59)
def calc_study(offset):
    start = ee.Date('2021-01-01').advance(offset, 'month')
    end = start.advance(1, 'month')
    rain = study_chirps.filterDate(start, end).sum()
    return rain.set('system:time_start', start.millis())
study_monthly = ee.ImageCollection.fromImages(study_months.map(calc_study))

# 3. Function to extract the average rainfall for Gode for each month
def extract_rainfall(image):
    # CHIRPS native resolution is ~5.5km (5566 meters)
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi_gode,
        scale=5566,
        maxPixels=1e9
    )
    # Format date as YYYY-MM for R
    date_str = ee.Date(image.get('system:time_start')).format('YYYY-MM')

    # Return a table row (Feature)
    return ee.Feature(None, {
        'Date': date_str,
        'Rainfall_mm': stats.get('precipitation')
    })

# 4. Apply the extraction to both collections and merge them
baseline_features = baseline_monthly.map(extract_rainfall)
study_features = study_monthly.map(extract_rainfall)
all_features = baseline_features.merge(study_features)

# 5. Export to Google Drive as a CSV
task = ee.batch.Export.table.toDrive(
    collection=all_features,
    description='Gode_Monthly_Rainfall_SPI',
    folder='EarthEngineExports',
    fileFormat='CSV',
    selectors=['Date', 'Rainfall_mm'] # Only export these two columns
)

task.start()
print("Export task started! Check your Google Drive in 1-2 minutes.")

Export task started! Check your Google Drive in 1-2 minutes.


In [11]:
import ee, geemap
ee.Initialize(project='irrigation-intelligence-pro')

# 1. Rebuild Gode boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
gode = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
gode = gode.filter(ee.Filter.eq('GAUL1_NAME', 'Somali'))
gode = gode.filter(ee.Filter.eq('GAUL2_NAME', 'Shabelle'))
aoi_gode = gode.geometry()

# 2. Load Land Cover (ESA WorldCover 2021)
land_cover = ee.ImageCollection('ESA/WorldCover/v200').first().clip(aoi_gode)

# 3. Load Soil Texture (OpenLandMap - Topsoil Sand Fraction)
# (We use sand fraction as a proxy for soil type)
soil_sand = ee.Image('OpenLandMap/SOL/SoilTex-Grids/USDA/1x1/0-30cm').select('b0').clip(aoi_gode)

# 4. Show them on a map to verify
m = geemap.Map()
m.centerObject(aoi_gode, 9)

# Land cover visualization (Standard ESA colors)
lc_vis = {'min': 10, 'max': 100, 'palette': ['006400', 'ffbb22', 'ffff4c', 'f096ff', 'fa0000', 'b4b4b4', 'f0f0f0', '0064c8', '0096a0', '00cf75', 'fae6a0']}
m.addLayer(land_cover, lc_vis, 'Land Cover')
m.addLayer(soil_sand, {'min': 0, 'max': 100, 'palette': ['red', 'yellow', 'green']}, 'Soil Sand %')
m

EEException: Image.load: Image asset 'OpenLandMap/SOL/SoilTex-Grids/USDA/1x1/0-30cm' not found (does not exist or caller does not have access).

In [12]:
import ee, geemap
ee.Initialize(project='irrigation-intelligence-pro')

# 1. Rebuild Gode boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
gode = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
gode = gode.filter(ee.Filter.eq('GAUL1_NAME', 'Somali'))
gode = gode.filter(ee.Filter.eq('GAUL2_NAME', 'Shabelle'))
aoi_gode = gode.geometry()

# 2. Load Land Cover (ESA WorldCover 2021) - unchanged
land_cover = ee.ImageCollection('ESA/WorldCover/v200').first().clip(aoi_gode)

# 3. Load Soil Texture with the CORRECT asset ID and select the topsoil band
soil_texture = ee.Image('OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02').select('b0').clip(aoi_gode)

# 4. Show them on a map
m = geemap.Map()
m.centerObject(aoi_gode, 9)

# Land cover visualization
lc_vis = {'min': 10, 'max': 100, 'palette': ['006400', 'ffbb22', 'ffff4c', 'f096ff', 'fa0000', 'b4b4b4', 'f0f0f0', '0064c8', '0096a0', '00cf75', 'fae6a0']}
m.addLayer(land_cover, lc_vis, 'Land Cover')

# Soil texture visualization (USDA classes)
# The dataset has 12 classes; we use a generic palette for display
soil_vis = {
    'min': 1, 'max': 12,
    'palette': ['d5c36b', 'b96947', '9d3706', 'ae868f', 'f86714', '46d143',
                '368f20', '3e5a14', 'ffd557', 'fff72e', 'ff5a9d', 'ff005b']
}
m.addLayer(soil_texture, soil_vis, 'Soil Texture (0cm)')

m

Map(center=[5.797427055226406, 43.659140145922386], controls=(WidgetControl(options=['position', 'transparent_…

In [13]:
import ee, geemap
ee.Initialize(project='irrigation-intelligence-pro')

# 1. Boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
gode = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia')) \
           .filter(ee.Filter.eq('GAUL1_NAME', 'Somali')) \
           .filter(ee.Filter.eq('GAUL2_NAME', 'Shabelle'))
aoi_gode = gode.geometry()

# 2. Data
land_cover = ee.ImageCollection('ESA/WorldCover/v200').first().clip(aoi_gode)
soil_texture = ee.Image('OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02').select('b0').clip(aoi_gode)

# 3. Map (Removes the blocked background)
m = geemap.Map()
m.remove(m.layers[0]) # <--- This removes the blocked background
m.centerObject(aoi_gode, 9)

lc_vis = {'min': 10, 'max': 100, 'palette': ['006400', 'ffbb22', 'ffff4c', 'f096ff', 'fa0000', 'b4b4b4', 'f0f0f0', '0064c8', '0096a0', '00cf75', 'fae6a0']}
soil_vis = {'min': 1, 'max': 12, 'palette': ['d5c36b', 'b96947', '9d3706', 'ae868f', 'f86714', '46d143', '368f20', '3e5a14', 'ffd557', 'fff72e', 'ff5a9d', 'ff005b']}

m.addLayer(land_cover, lc_vis, 'Land Cover')
m.addLayer(soil_texture, soil_vis, 'Soil Texture (0cm)')
m

Map(center=[5.797427055226406, 43.659140145922386], controls=(WidgetControl(options=['position', 'transparent_…

In [14]:
import ee, geemap
ee.Initialize(project='irrigation-intelligence-pro')

# 1. Boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
gode = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia')) \
           .filter(ee.Filter.eq('GAUL1_NAME', 'Somali')) \
           .filter(ee.Filter.eq('GAUL2_NAME', 'Shabelle'))
aoi_gode = gode.geometry()

# 2. Data
land_cover = ee.ImageCollection('ESA/WorldCover/v200').first().clip(aoi_gode)

# 3. Create Map (Removing blocked background)
m = geemap.Map()
m.remove(m.layers[0])
m.centerObject(aoi_gode, 9)

# 4. Exact colors for ESA Land Cover
lc_vis = {
    'min': 10, 'max': 100,
    'palette': ['006400', 'ffbb22', 'ffff4c', 'f096ff', 'fa0000', 'b4b4b4', 'f0f0f0', '0064c8', '0096a0', '00cf75', 'fae6a0']
}
m.addLayer(land_cover, lc_vis, 'Land Cover')

# 5. Legend using exact same hex codes
legend_dict = {
    '10 - Tree cover': '006400',
    '20 - Shrubland': 'ffbb22',
    '30 - Grassland': 'ffff4c',
    '40 - Cropland': 'f096ff',
    '50 - Built-up': 'fa0000',
    '60 - Bare / sparse': 'b4b4b4',
    '80 - Water bodies': '0064c8',
    '90 - Wetlands': '0096a0'
}
m.add_legend(title='ESA Land Cover Classes', legend_dict=legend_dict, position='bottomright')

m

Map(center=[5.797427055226406, 43.659140145922386], controls=(WidgetControl(options=['position', 'transparent_…

In [16]:
import ee
ee.Initialize(project='irrigation-intelligence-pro')

# 1. Rebuild Boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
gode = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia')) \
           .filter(ee.Filter.eq('GAUL1_NAME', 'Somali')) \
           .filter(ee.Filter.eq('GAUL2_NAME', 'Shabelle'))
aoi_gode = gode.geometry()

# 2. Load Land Cover and assign Curve Numbers (CN)
# ESA classes: 10=Tree, 20=Shrub, 30=Grass, 40=Crop, 50=Built, 60=Bare, 80=Water
land_cover = ee.ImageCollection('ESA/WorldCover/v200').first().clip(aoi_gode)
# Remap land cover classes to approximate CN values for semi-arid regions
cn_image = land_cover.remap(
    [10, 20, 30, 40, 50, 60, 80],  # Input classes
    [55, 70, 75, 80, 90, 85, 98]   # Corresponding CN values
).rename('CN')

# 3. Load Rainfall (CHIRPS) for 2021-2025
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
study_chirps = chirps.filterBounds(aoi_gode).filterDate('2021-01-01', '2025-12-31')

# 4. Sequence for monthly aggregation
start_date = ee.Date('2021-01-01')
months = ee.List.sequence(0, 59)

# 5. The Function to calculate Monthly Runoff (SCS-CN)
def calculate_monthly_flood(month_offset):
    # Get the monthly rainfall sum (P)
    current_month_start = start_date.advance(month_offset, 'month')
    current_month_end = current_month_start.advance(1, 'month')
    monthly_rain = study_chirps.filterDate(current_month_start, current_month_end).sum()

    # The Math: S = (25400 / CN) - 254
    S = cn_image.expression('(25400 / CN) - 254', {'CN': cn_image})

    # The Math: Q = ((P - 0.2S)^2) / (P + 0.8S)
    # We use .max(0) because runoff cannot be negative (if rain is less than 0.2S, no runoff)
    Q = monthly_rain.expression(
        '((P - 0.2 * S)**2) / (P + 0.8 * S)', {
            'P': monthly_rain,
            'S': S
        }
    ).max(0).rename('Runoff')

    return Q.set('system:time_start', current_month_start.millis())

# 6. Map the function to all 60 months
monthly_flood_col = ee.ImageCollection.fromImages(months.map(calculate_monthly_flood))

# 7. Extract the mean runoff for Gode for each month
def extract_flood(image):
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi_gode,
        scale=5566,
        maxPixels=1e9
    )
    date_str = ee.Date(image.get('system:time_start')).format('YYYY-MM')
    return ee.Feature(None, {
        'Date': date_str,
        'Flood_Runoff_mm': stats.get('Runoff')
    })

flood_features = monthly_flood_col.map(extract_flood)

# 8. Export to Google Drive
task = ee.batch.Export.table.toDrive(
    collection=flood_features,
    description='Gode_Monthly_Flood_SCS',
    folder='EarthEngineExports',
    fileFormat='CSV',
    selectors=['Date', 'Flood_Runoff_mm']
)
task.start()
print("Flood export task started! Check your Google Drive in 1-2 minutes for Gode_Monthly_Flood_SCS.csv")

Flood export task started! Check your Google Drive in 1-2 minutes for Gode_Monthly_Flood_SCS.csv


In [17]:
import ee
ee.Initialize(project='irrigation-intelligence-pro')

# 1. Rebuild Boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
gode = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia')) \
           .filter(ee.Filter.eq('GAUL1_NAME', 'Somali')) \
           .filter(ee.Filter.eq('GAUL2_NAME', 'Shabelle'))
aoi_gode = gode.geometry()

# 2. Load ERA5-Land Daily Data
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')

# 3. Filter by Boundary and Date (2021-2025)
era5_filtered = era5.filterBounds(aoi_gode).filterDate('2021-01-01', '2025-12-31')

# 4. Count the days
count = era5_filtered.size().getInfo()
print(f"Number of daily ERA5-Land climate images: {count}")

# 5. Check what bands are available
bands = era5_filtered.first().bandNames().getInfo()
print("Available climate variables:")
for b in bands:
    print("  -", b)

Number of daily ERA5-Land climate images: 1825
Available climate variables:
  - dewpoint_temperature_2m
  - temperature_2m
  - skin_temperature
  - soil_temperature_level_1
  - soil_temperature_level_2
  - soil_temperature_level_3
  - soil_temperature_level_4
  - lake_bottom_temperature
  - lake_ice_depth
  - lake_ice_temperature
  - lake_mix_layer_depth
  - lake_mix_layer_temperature
  - lake_shape_factor
  - lake_total_layer_temperature
  - snow_albedo
  - snow_cover
  - snow_density
  - snow_depth
  - snow_depth_water_equivalent
  - snowfall_sum
  - snowmelt_sum
  - temperature_of_snow_layer
  - skin_reservoir_content
  - volumetric_soil_water_layer_1
  - volumetric_soil_water_layer_2
  - volumetric_soil_water_layer_3
  - volumetric_soil_water_layer_4
  - forecast_albedo
  - surface_latent_heat_flux_sum
  - surface_net_solar_radiation_sum
  - surface_net_thermal_radiation_sum
  - surface_sensible_heat_flux_sum
  - surface_solar_radiation_downwards_sum
  - surface_thermal_radiation_d

In [18]:
import ee
ee.Initialize(project='irrigation-intelligence-pro')

# 1. Rebuild Boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level2')
gode = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia')) \
           .filter(ee.Filter.eq('GAUL1_NAME', 'Somali')) \
           .filter(ee.Filter.eq('GAUL2_NAME', 'Shabelle'))
aoi_gode = gode.geometry()

# 2. Load ERA5 Data
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR') \
    .filterBounds(aoi_gode) \
    .filterDate('2021-01-01', '2025-12-31')

# 3. The Exact FAO-56 Penman-Monteith Formula
def calculate_et0(image):
    # Temperature in Celsius (from Kelvin)
    t = image.select('temperature_2m').subtract(273.15)
    tdew = image.select('dewpoint_temperature_2m').subtract(273.15)

    # Wind speed at 2m (converting from 10m)
    u10 = image.select('u_component_of_wind_10m')
    v10 = image.select('v_component_of_wind_10m')
    wind10 = u10.pow(2).add(v10.pow(2)).sqrt()
    u2 = wind10.multiply(0.748)

    # Net Radiation (J/m2 to MJ/m2)
    rn = image.select('surface_net_solar_radiation_sum') \
              .add(image.select('surface_net_thermal_radiation_sum')).divide(1e6)

    # Vapor Pressures (kPa)
    es = t.multiply(17.27).divide(t.add(237.3)).exp().multiply(0.6108)
    ea = tdew.multiply(17.27).divide(tdew.add(237.3)).exp().multiply(0.6108)

    # Slope of vapor pressure and Psychrometric constant
    delta = es.multiply(4098).divide(t.add(237.3).pow(2))
    p = image.select('surface_pressure').divide(1000)
    gamma = p.multiply(0.000665)

    # FAO-56 Equation
    num = delta.multiply(rn).multiply(0.408).add(gamma.multiply(900).divide(t.add(273)).multiply(u2).multiply(es.subtract(ea)))
    den = delta.add(gamma.multiply(u2.multiply(0.34).add(1)))

    et0 = num.divide(den).rename('ET0_mm')
    # Max 0 because ET cannot be negative
    return et0.max(0).copyProperties(image, ['system:time_start'])

# Apply formula to all 1825 days
daily_et0 = era5.map(calculate_et0)

# 4. Sum daily ET0 into Monthly Totals
start_date = ee.Date('2021-01-01')
months = ee.List.sequence(0, 59)

def sum_monthly_et0(month_offset):
    start = start_date.advance(month_offset, 'month')
    end = start.advance(1, 'month')
    monthly_et = daily_et0.filterDate(start, end).sum().rename('ET0_mm')
    return monthly_et.set('system:time_start', start.millis())

monthly_et_col = ee.ImageCollection.fromImages(months.map(sum_monthly_et0))

# 5. Extract for Gode
def extract_et(image):
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi_gode,
        scale=11132, # ERA5-Land native resolution is ~11km
        maxPixels=1e9
    )
    date_str = ee.Date(image.get('system:time_start')).format('YYYY-MM')
    return ee.Feature(None, {
        'Date': date_str,
        'ET0_mm': stats.get('ET0_mm')
    })

et_features = monthly_et_col.map(extract_et)

# 6. Export to Drive
task = ee.batch.Export.table.toDrive(
    collection=et_features,
    description='Gode_Monthly_ET0_FAO56',
    folder='EarthEngineExports',
    fileFormat='CSV',
    selectors=['Date', 'ET0_mm']
)
task.start()
print("ET0 export task started! Check your Google Drive for Gode_Monthly_ET0_FAO56.csv")

ET0 export task started! Check your Google Drive for Gode_Monthly_ET0_FAO56.csv
